# 예제 franka_ex10: FR3 OMPL 플래너 비교

같은 joint 목표를 5 개 OMPL 플래너에게 시켜 보고, 계획 시간 / 포인트 수 / EE 경로 모양을
비교한다. RViz 에는 5 색 LINE_STRIP 이 **동시에** 누적되어 한눈에 비교.

## 이 노트북이 강조하는 것 — `planner_id` 와 누적 시각화

ex03~09 의 모든 plan 은 `planner_id` 를 비워두어 MoveIt 의 **기본 플래너 (RRTConnect)** 를
썼다. ex10 은 처음으로 `MotionPlanRequest.planner_id` 를 명시해 플래너를 골라잡고,
같은 목표를 향한 5 개 결과를 누적해 비교한다.

| 플래너 | 특성 |
|---|---|
| **RRTConnect** | 양방향 RRT — 빠르고 안정적 (기본) |
| **RRT** | 단방향 RRT — 단순, 느릴 수 있음 |
| **PRM** | Probabilistic Roadmap — 다중 쿼리에 유리 |
| **EST** | Expansive Space Trees — 좁은 공간에서 강함 |
| **KPIECE** | Kinematic Planning by Interior-Exterior Cell Exploration — 고차원 공간에 효과적 |

## 복잡한 경로 환경

빈 공간에서는 모든 플래너가 비슷한 직선 경로를 찾기 때문에 차이가 안 드러난다.
이 노트북은 ex09 의 `make_box` / `add_object` 를 재사용해 좌측 어깨 영역에 **두꺼운
top_bar (z=0.45~0.85) + bottom_bar (z=0.15~0.25)** 를 만든다. top_bar 가 위쪽 자유
공간까지 막아서 — plan 은 통로 (z=0.25~0.45) 사이 또는 박스 옆 (y>0.40) 으로만 통과
가능하다. ready (정면 y=0) 에서 target_joints (좌측 통로 안 EE) 로 가는 동안 EE 와
robot link 들이 이 좁은 영역을 통과해야 하므로:

- RRTConnect 는 보통 빠르고 짧은 경로
- RRT 는 단방향이라 시간 ↑, 더 굴곡진 경로
- PRM 은 첫 호출은 느리지만 같은 환경 다중 쿼리에 강함
- EST / KPIECE 는 좁은 공간 탐색에 특화 — 다른 모양의 우회 경로

## 노트북 구성
1. **로봇 상수 + 플래너 목록**
2. **핵심 — `planner_id` 비교 워크플로** ← 이 노트북의 본질
3. **핵심을 쓰기 위한 설정** — ROS init, 클라이언트, FK, SRDF, Pose / MoveGroup / 충돌 객체 / 마커
4. **시나리오** — 통로 환경 → 5 개 플래너 비교 → 결과표

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 에서:
- **Planning Scene Display** 가 켜져 있어야 통로 막대가 보인다
- **`MarkerArray` Display** 추가, Topic `/planner_paths` — 5 색 EE 경로
- Fixed Frame `fr3_link0`

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab franka_ex10_multi_planner.ipynb
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 로봇 상수 + 플래너 목록

In [1]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC      = '/planner_paths'

# 색은 (r, g, b) 튜플로 — ColorRGBA 객체는 마커 만들 때 변환
PLANNERS = [
    {'id': 'RRTConnect', 'rgb': (1.0, 0.2, 0.2),
     'desc': '양방향 RRT — 빠르고 안정적 (기본)'},
    {'id': 'RRT',        'rgb': (0.2, 0.8, 0.2),
     'desc': '단방향 RRT — 단순, 느릴 수 있음'},
    {'id': 'PRM',        'rgb': (0.2, 0.4, 1.0),
     'desc': 'Probabilistic Roadmap — 다중 쿼리에 유리'},
    {'id': 'EST',        'rgb': (1.0, 0.6, 0.0),
     'desc': 'Expansive Space Trees — 좁은 공간에서 강함'},
    {'id': 'KPIECE',     'rgb': (0.7, 0.2, 0.9),
     'desc': 'KPIECE — 투영 기반, 고차원 공간에 효과적'},
]

## 2. 핵심 — `planner_id` 비교 워크플로

이 노트북에서 가장 먼저 정의해야 하는 함수들.

- `plan_with_planner()` — `planner_id` 명시 + 시간 측정
- `add_planner_path()` — 플래너 별 색 LINE_STRIP + 끝점 sphere + 라벨 (누적)
- `compare_planner()` — 한 플래너에 대해 reset → plan → mark → execute → 결과 측정
- `print_comparison_table()` — 결과 비교표 출력

함수들은 setup 셀에서 만드는 객체 (`node`, `move_client`, `_markers` 등) 와 헬퍼
(`make_plan_request`, `send_move_goal`, `trajectory_to_ee_path`, `execute_trajectory`,
`go_to_joint_goal`, `make_joint_constraints`) 를 globals 로 참조 — 정의 시점엔 lookup
하지 않으므로 객체가 아직 없어도 OK. 호출은 4 절(시나리오) 에서.

### 2-1. 핵심에 필요한 import

In [2]:
import time
from std_msgs.msg import ColorRGBA
from visualization_msgs.msg import Marker
from geometry_msgs.msg import Point, Vector3
from moveit_msgs.msg import MoveItErrorCodes

### 2-2. `plan_with_planner()` — `planner_id` 명시 + 시간 측정

`MotionPlanRequest.planner_id` 에 OMPL 플래너 이름 (`'RRTConnect'`, `'EST'` 등) 을 넣고
`plan_only=True` 로 trajectory 만 받는다. wall-clock 으로 호출 시간 측정.

In [3]:
def plan_with_planner(joint_values: dict, planner_id: str,
                      vel: float = 0.3, plan_time: float = 10.0):
    '''planner_id 를 명시한 plan-only + 경과 시간 반환.'''
    req = make_plan_request(vel=vel, acc=vel, plan_time=plan_time,
                            planner_id=planner_id)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    t0 = time.time()
    code_val, traj = send_move_goal(req, plan_only=True)
    elapsed = time.time() - t0
    return code_val == MoveItErrorCodes.SUCCESS, traj, elapsed

### 2-3. `add_planner_path()` — 색별 누적 LINE_STRIP + 끝점 sphere + 라벨

플래너 마다 ns 가 다르므로 (`path_RRT`, `path_PRM` 등) 같이 보인다.
같은 ns / id 의 이전 마커는 새로 publish 시 덮어써져, 노트북 재실행해도 누적되지 않는다.

라벨은 trajectory 의 중간점 위에 띄우되 플래너 인덱스만큼 z 오프셋을 줘서 라벨끼리
겹치지 않게 한다.

In [4]:
def _rgba(rgb, a: float = 0.95) -> ColorRGBA:
    return ColorRGBA(r=rgb[0], g=rgb[1], b=rgb[2], a=a)


def add_planner_path(planner_id: str, ee_points, planner_idx: int,
                     rgb, width: float = 0.005) -> None:
    if not ee_points:
        return
    color = _rgba(rgb)
    stamp = node.get_clock().now().to_msg()

    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = f'path_{planner_id}'
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = width
    line.color = color
    line.points = [Point(x=p[0], y=p[1], z=p[2]) for p in ee_points]

    end = Marker()
    end.header.frame_id = REFERENCE_FRAME
    end.header.stamp = stamp
    end.ns = f'end_{planner_id}'
    end.id = 0
    end.type = Marker.SPHERE
    end.action = Marker.ADD
    end.pose.position = Point(x=ee_points[-1][0],
                              y=ee_points[-1][1],
                              z=ee_points[-1][2])
    end.pose.orientation.w = 1.0
    end.scale = Vector3(x=0.025, y=0.025, z=0.025)
    end.color = color

    label = Marker()
    label.header.frame_id = REFERENCE_FRAME
    label.header.stamp = stamp
    label.ns = f'label_{planner_id}'
    label.id = 0
    label.type = Marker.TEXT_VIEW_FACING
    label.action = Marker.ADD
    mid = ee_points[len(ee_points) // 2]
    label.pose.position = Point(
        x=mid[0],
        y=mid[1],
        z=mid[2] + 0.06 + 0.04 * planner_idx,
    )
    label.pose.orientation.w = 1.0
    label.scale.z = 0.04
    label.color = color
    label.text = planner_id

    new_keys = {(m.ns, m.id) for m in (line, end, label)}
    _markers.markers = [m for m in _markers.markers
                        if (m.ns, m.id) not in new_keys]
    _markers.markers.extend([line, end, label])
    marker_pub.publish(_markers)

### 2-4. `compare_planner()` — 한 플래너 비교 실행

한 플래너에 대해 다음을 한 호출로:

1. **reset** — `go_to_joint_goal(ready_target)` 으로 ready 자세 복귀 (RRTConnect 사용,
   비교 대상 아님)
2. **plan** — `plan_with_planner()` 로 trajectory + 시간 측정
3. **mark** — FK 로 EE 경로 환산 → `add_planner_path()`
4. **execute** — `execute_trajectory()` 로 실제 이동
5. **결과 dict 반환** — `{planner, success, plan_time, n_pts}`

In [5]:
def compare_planner(planner_id: str, planner_idx: int, rgb,
                    target: dict, plan_time: float = 10.0) -> dict:
    # 1. reset (RRTConnect 사용 — 비교 대상 아님)
    go_to_joint_goal(ready_target, vel=0.4, acc=0.4)
    time.sleep(0.3)

    # 2. 비교 대상 플래너로 plan-only
    ok, traj, elapsed = plan_with_planner(
        target, planner_id, vel=0.3, plan_time=plan_time,
    )

    n_pts = 0
    if ok and traj is not None:
        n_pts = len(traj.joint_trajectory.points)
        # 3. FK 로 EE 경로 추출 → RViz 누적
        ee_pts = trajectory_to_ee_path(traj)
        if ee_pts:
            add_planner_path(planner_id, ee_pts, planner_idx, rgb)
        # 4. 실제 실행
        execute_trajectory(traj)
        node.get_logger().info(
            f'  → {planner_id}: OK ({elapsed:.2f}s, {n_pts} pts)'
        )
    else:
        node.get_logger().warn(
            f'  → {planner_id}: FAIL ({elapsed:.2f}s)'
        )

    return {'planner': planner_id, 'success': ok,
            'plan_time': elapsed, 'n_pts': n_pts}

### 2-5. `print_comparison_table()` — 결과 비교표 출력

In [6]:
def print_comparison_table(results) -> None:
    node.get_logger().info('=' * 60)
    node.get_logger().info('  플래너 비교 결과')
    node.get_logger().info('=' * 60)
    node.get_logger().info(f'  {"Planner":<12} {"Status":>6}  {"Time":>8}  {"Points":>6}')
    node.get_logger().info('-' * 60)
    for r in results:
        status = 'OK' if r['success'] else 'FAIL'
        node.get_logger().info(
            f"  {r['planner']:<12} {status:>6}  {r['plan_time']:>6.2f}s  {r['n_pts']:>6}"
        )
    node.get_logger().info('=' * 60)

## 3. 핵심을 쓰기 위한 설정

위 핵심 함수들이 참조하는 객체와 보조 헬퍼.

- ROS 2 초기화 / 노드 / 액션·서비스 클라이언트 (`apply_planning_scene` + `compute_fk` 포함)
- 서버·서비스·`/joint_states` 준비 대기
- SRDF `ready` 자세
- Pose 헬퍼
- MoveGroup 빌딩블록 (Constraints / `MotionPlanRequest`)
- ex07 패턴 — `send_move_goal()` / `plan_to_joint_goal()` / `go_to_joint_goal()` / `trajectory_to_ee_path()` / `execute_trajectory()`
- ex09 패턴 — 충돌 객체 헬퍼 (`make_box` / `add_object` / `clear_all`)
- 마커 컨테이너

### 3-1. ROS 2 초기화 + 노드 + 클라이언트

In [7]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from moveit_msgs.srv import GetPositionFK, ApplyPlanningScene
from visualization_msgs.msg import MarkerArray

try:
    rclpy.init()
except RuntimeError:
    pass

node = Node(
    'franka_ex10_planners_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client    = ActionClient(node, MoveGroup, 'move_action')
execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
fk_client      = node.create_client(GetPositionFK, 'compute_fk')
scene_client   = node.create_client(ApplyPlanningScene, 'apply_planning_scene')
marker_pub     = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex10 노트북 노드 생성 완료 ===')

[INFO] [1778407492.071902703] [franka_ex10_planners_demo]: === franka_ex10 노트북 노드 생성 완료 ===


True

### 3-2. 액션 / 서비스 / `/joint_states` 준비 대기

In [8]:
def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    if not execute_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')
    if not fk_client.wait_for_service(timeout_sec=timeout_sec):
        raise RuntimeError('compute_fk 서비스 연결 실패')
    if not scene_client.wait_for_service(timeout_sec=timeout_sec):
        raise RuntimeError('apply_planning_scene 서비스 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info(
        'move/execute action + compute_fk + apply_planning_scene + /joint_states 준비됨'
    )

wait_for_ready()

[INFO] [1778407492.333278005] [franka_ex10_planners_demo]: move/execute action + compute_fk + apply_planning_scene + /joint_states 준비됨


### 3-3. SRDF 에서 `ready` 자세

In [9]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

[INFO] [1778407492.349337490] [franka_ex10_planners_demo]: ready: {'fr3_joint1': 0.0, 'fr3_joint2': -0.7853981633974483, 'fr3_joint3': 0.0, 'fr3_joint4': -2.356194490192345, 'fr3_joint5': 0.0, 'fr3_joint6': 1.5707963267948966, 'fr3_joint7': 0.7853981633974483}


True

### 3-4. Pose 헬퍼

In [10]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

### 3-5. MoveGroup 빌딩블록 — Constraints / `MotionPlanRequest`

In [11]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, RobotState,
)
from shape_msgs.msg import SolidPrimitive

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

### 3-6. `send_move_goal()` + plan-only / plan+execute / FK / execute

ex07 패턴 그대로. `compare_planner()` 가 plan-only (시간 측정용) 와 plan+execute
(reset 용) 두 모드를 모두 사용하므로 두 변형 다 정의.

In [12]:
def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only,
        replan=not plan_only,
        replan_attempts=3 if not plan_only else 0,
    )
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory


def go_to_joint_goal(joint_values: dict, vel: float = 0.3,
                     acc: float = 0.3) -> bool:
    '''reset 용 — plan+execute 통합. EE 경로 시각화 안 함 (누적 마커 어지러워지지 않게).'''
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'reset 실패 error_code={code_val}')
    return ok


def trajectory_to_ee_path(trajectory, max_points: int = 60):
    jt = trajectory.joint_trajectory
    total = len(jt.points)
    if total == 0:
        return []
    step = max(1, total // max_points)
    indices = list(range(0, total, step))
    if indices[-1] != total - 1:
        indices.append(total - 1)
    pts = []
    for idx in indices:
        req = GetPositionFK.Request()
        req.header.frame_id = REFERENCE_FRAME
        req.fk_link_names = [END_EFFECTOR_LINK]
        rs = RobotState()
        rs.joint_state.name = list(jt.joint_names)
        rs.joint_state.position = list(jt.points[idx].positions)
        req.robot_state = rs
        fut = fk_client.call_async(req)
        rclpy.spin_until_future_complete(node, fut)
        resp = fut.result()
        if resp and resp.error_code.val == MoveItErrorCodes.SUCCESS and resp.pose_stamped:
            p = resp.pose_stamped[0].pose.position
            pts.append((p.x, p.y, p.z))
    return pts


def execute_trajectory(trajectory) -> bool:
    g = ExecuteTrajectory.Goal()
    g.trajectory = trajectory
    sf = execute_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    return rf.result().result.error_code.val == MoveItErrorCodes.SUCCESS

### 3-7. 충돌 객체 헬퍼 (ex09 재사용)

이 노트북에서는 통로 환경 — 위 / 아래 가로 막대 두 개 — 만 만든다.
ex09 의 `make_box` / `apply_diff` / `add_object` / `clear_all` 그대로.

In [13]:
from moveit_msgs.msg import PlanningScene, CollisionObject

def make_box(object_id: str, position, dimensions,
             frame_id: str = None) -> CollisionObject:
    co = CollisionObject()
    co.header.frame_id = frame_id or REFERENCE_FRAME
    co.id = object_id
    co.operation = CollisionObject.ADD
    box = SolidPrimitive()
    box.type = SolidPrimitive.BOX
    box.dimensions = list(dimensions)
    co.primitives.append(box)
    pose = Pose()
    pose.position = Point(x=position[0], y=position[1], z=position[2])
    pose.orientation = Quaternion(x=0.0, y=0.0, z=0.0, w=1.0)
    co.primitive_poses.append(pose)
    return co


def apply_diff(world_objects=None) -> bool:
    scene = PlanningScene()
    scene.is_diff = True
    if world_objects:
        scene.world.collision_objects = list(world_objects)
    req = ApplyPlanningScene.Request()
    req.scene = scene
    fut = scene_client.call_async(req)
    rclpy.spin_until_future_complete(node, fut)
    return fut.result().success


def add_object(co: CollisionObject) -> bool:
    return apply_diff([co])


def clear_all() -> bool:
    co = CollisionObject()
    co.header.frame_id = REFERENCE_FRAME
    co.id = ''
    co.operation = CollisionObject.REMOVE
    return apply_diff([co])

### 3-8. 마커 컨테이너

In [14]:
_markers = MarkerArray()

## 4. 시나리오 — 통로 환경에서 5 개 플래너 비교

1. ready 자세로 시작
2. 통로 환경 (위 / 아래 가로 막대) 추가
3. 큰 자세 변화 목표 정의 — joint1=+90°, joint4=-150° 등 7-DOF 모두 변함
4. 5 개 플래너 순회 — 각각 reset → plan(시간 측정) → 누적 mark → execute
5. 비교표 출력
6. clear_all + ready 복귀

### 4-1. ready 자세로 초기화

In [15]:
node.get_logger().info('--- ready 자세로 초기 이동 ---')
go_to_joint_goal(ready_target, vel=0.4, acc=0.4)
time.sleep(1.0)

[INFO] [1778407495.181345956] [franka_ex10_planners_demo]: --- ready 자세로 초기 이동 ---


### 4-2. 통로 환경 — 좌측 어깨 영역, 박스 위쪽 + 옆쪽 봉쇄

박스를 robot 본체 가까이 두면 ready 자세 link 와 충돌하고, 너무 멀리 두면 EE 동선과
무관해진다. **`y=+0.35` 좌측 어깨~깊이 영역** 에 두면 ready 자세 (정면 y=0) 와 안 겹치면서도
target 자세 (좌측으로 팔 뻗음) 의 link 동선을 가로지른다.

이전 버전 시각화에서 plan 들이 박스 위쪽 또는 옆 (y>0.40) 으로 빠져나가 통로 통과 효과가
약했다. 두 단계로 봉쇄:

1. **위쪽 (z) 봉쇄** — `top_bar` z 두께를 `0.40` 으로 → z=0.45~0.85 차단
2. **옆쪽 (y) 봉쇄** — `top_bar` / `bottom_bar` y dim 을 `0.30` 으로, y_max 를 +0.40 → +0.50
   → plan 이 박스 옆으로 빠져나가려면 y>0.50 이어야 (target F link5 의 y=0.554 보다 좁은 통로)

| 이름 | 중심 (x, y, z) | 크기 (x, y, z) | 효과 |
|---|---|---|---|
| `top_bar` | (0.40, **0.35**, 0.65) | (0.30, **0.30**, 0.40) | x=0.25~0.55, y=0.20~**0.50**, z=0.45~0.85 |
| `bottom_bar` | (0.40, **0.35**, 0.20) | (0.30, **0.30**, 0.10) | x=0.25~0.55, y=0.20~**0.50**, z=0.15~0.25 |

FK 검증:
- ready / target_joints 양 끝점 모두 박스 영역 밖 → collision-free
- 직선 보간 t=0.5 에서 link5/6/7/flange/hand_tcp 5 개 모두 TOP_BAR 안 → plan 우회 필요
- target F link5 의 y=0.554 > 박스 y_max=0.50 — 박스 옆 좁은 틈이 남아 EE 도달 가능

In [16]:
top_bar = make_box('top_bar',
                   position=(0.40, 0.35, 0.65),
                   dimensions=(0.30, 0.30, 0.40))   # y/z 모두 확장
bottom_bar = make_box('bottom_bar',
                      position=(0.40, 0.35, 0.20),
                      dimensions=(0.30, 0.30, 0.10))   # y 확장
add_object(top_bar)
add_object(bottom_bar)
node.get_logger().info('  좌측 어깨 통로 환경 (top y/z 봉쇄, 통로 z=0.25~0.45 / y=0.20~0.50)')
time.sleep(1.0)

[INFO] [1778407496.368422062] [franka_ex10_planners_demo]:   좌측 어깨 통로 환경 (top y/z 봉쇄, 통로 z=0.25~0.45 / y=0.20~0.50)


### 4-3. 큰 자세 변화 목표 정의

forward kinematics 검증 결과 아래 joint 조합이 본 환경에 적합:

- **target EE = (0.00, +0.584, +0.371)** — y 박스 너머 좌측, z 가 통로 z 범위 (0.25~0.45) 안
- ready → target joint 직선 보간 t=0.5 에서 EE 가 (+0.318, +0.318, +0.471) — **TOP_BAR 안**
- 즉 단순 직선 보간으로는 박스 충돌 → plan 이 *반드시* 우회 trajectory 를 만들어야 함

joint2 를 +15° 로 어깨 약간 숙이고 (이전 -60° 는 어깨를 위로 들어 EE 가 머리 위로 가버림),
joint6=+90° / joint7=+45° 로 손목을 돌려 EE 가 좌측 통로 z 범위에 정확히 안착.

plan_time 은 KPIECE / EST 가 좁은 공간에서 시간을 좀 더 쓸 수 있도록 15 초로.

In [17]:
target_joints = {
    'fr3_joint1':  math.radians(90),    # 좌측 90° 회전 → EE +y 방향
    'fr3_joint2':  math.radians(15),    # 어깨 약간 숙이기 (EE 가 머리 위로 안 가게)
    'fr3_joint3':  math.radians(0),
    'fr3_joint4':  math.radians(-90),   # 팔꿈치 직각
    'fr3_joint5':  math.radians(0),
    'fr3_joint6':  math.radians(90),    # 손목 회전 → EE z 를 통로 안으로
    'fr3_joint7':  math.radians(45),    # 손목 마지막 회전
}
# FK 검증: EE = (0.00, +0.584, +0.371), 모든 link 가 박스 영역 밖 (collision-free).
# ready → target 직선 보간 t=0.5 에서 EE (+0.32, +0.32, +0.47) → TOP_BAR 충돌 → plan 우회 강제.

PLAN_TIME = 15.0

### 4-4. 5 개 플래너 비교 실행

루프 한 번이 한 플래너. RViz 에는 5 색 LINE_STRIP 이 차례로 누적되어 마지막에는 모두
한 화면에 보인다. 같은 ns 의 이전 마커는 새 publish 가 덮어쓰므로 셀을 재실행해도
마커가 두 겹으로 찍히지 않는다.

In [18]:
results = []
for i, planner in enumerate(PLANNERS):
    node.get_logger().info(
        f"\n--- 플래너 {i+1}/{len(PLANNERS)}: {planner['id']} ---"
    )
    node.get_logger().info(f"  설명: {planner['desc']}")
    r = compare_planner(planner['id'], i, planner['rgb'],
                        target_joints, plan_time=PLAN_TIME)
    results.append(r)
    time.sleep(0.5)

[INFO] [1778407503.896031080] [franka_ex10_planners_demo]: 
--- 플래너 1/5: RRTConnect ---
[INFO] [1778407503.896963674] [franka_ex10_planners_demo]:   설명: 양방향 RRT — 빠르고 안정적 (기본)
[INFO] [1778407511.918721520] [franka_ex10_planners_demo]:   → RRTConnect: OK (0.03s, 76 pts)
[INFO] [1778407512.420936824] [franka_ex10_planners_demo]: 
--- 플래너 2/5: RRT ---
[INFO] [1778407512.421692950] [franka_ex10_planners_demo]:   설명: 단방향 RRT — 단순, 느릴 수 있음
[INFO] [1778407523.997430269] [franka_ex10_planners_demo]:   → RRT: OK (0.02s, 70 pts)
[INFO] [1778407524.499430443] [franka_ex10_planners_demo]: 
--- 플래너 3/5: PRM ---
[INFO] [1778407524.500322953] [franka_ex10_planners_demo]:   설명: Probabilistic Roadmap — 다중 쿼리에 유리
[INFO] [1778407534.419768427] [franka_ex10_planners_demo]:   → PRM: OK (0.02s, 50 pts)
[INFO] [1778407534.921689710] [franka_ex10_planners_demo]: 
--- 플래너 4/5: EST ---
[INFO] [1778407534.922516207] [franka_ex10_planners_demo]:   설명: Expansive Space Trees — 좁은 공간에서 강함
[INFO] [1778407545.28950680

### 4-5. 비교표 출력

In [19]:
print_comparison_table(results)

[INFO] [1778407557.925378427] [franka_ex10_planners_demo]: ============================================================
[INFO] [1778407557.926315449] [franka_ex10_planners_demo]:   플래너 비교 결과
[INFO] [1778407557.926959447] [franka_ex10_planners_demo]: ============================================================
[INFO] [1778407557.927473092] [franka_ex10_planners_demo]:   Planner      Status      Time  Points
[INFO] [1778407557.928047991] [franka_ex10_planners_demo]: ------------------------------------------------------------
[INFO] [1778407557.928623742] [franka_ex10_planners_demo]:   RRTConnect       OK    0.03s      76
[INFO] [1778407557.929154999] [franka_ex10_planners_demo]:   RRT              OK    0.02s      70
[INFO] [1778407557.929676860] [franka_ex10_planners_demo]:   PRM              OK    0.02s      50
[INFO] [1778407557.930299848] [franka_ex10_planners_demo]:   EST              OK    0.02s      54
[INFO] [1778407557.931041668] [franka_ex10_planners_demo]:   KPIECE           

### 4-6. 통로 제거 + ready 복귀

In [21]:
go_to_joint_goal(ready_target, vel=0.4, acc=0.4)
clear_all()
node.get_logger().info('=== franka_ex10 완료 — 통로 제거됨 ===')

[INFO] [1778407567.200540567] [franka_ex10_planners_demo]: === franka_ex10 완료 — 통로 제거됨 ===


True

## 5. 정리

In [22]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass